In [1]:
# Cell 1: Authenticate to the Hub
!pip install --quiet huggingface_hub
from huggingface_hub import notebook_login
notebook_login()


In [2]:
# Cell 2: Core libraries
!pip install --quiet \
  transformers torch \
  sentence-transformers faiss-cpu \
  gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:

from collections import deque
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class MemoryManager:
    def __init__(self, embed_model_name="all-MiniLM-L6-v2"):
        self.ephemeral = deque(maxlen=3)
        self.embed = SentenceTransformer(embed_model_name)
        dim = self.embed.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(dim)
        self.texts = []

    def add_message(self, text: str):
        self.ephemeral.append(text)
        emb = self.embed.encode(text, convert_to_numpy=True)
        self.index.add(np.array([emb], dtype="float32"))
        self.texts.append(text)

    def retrieve(self, query: str, k: int = 5):
        if not self.texts:
            return []
        q_emb = self.embed.encode(query, convert_to_numpy=True)
        D, I = self.index.search(np.array([q_emb], dtype="float32"), k)
        return [self.texts[i] for i in I[0] if i < len(self.texts)]


In [4]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [5]:

session_memories = {}

def chat_with_memory(user_id: str, user_message: str) -> str:
    mem = session_memories.setdefault(user_id, MemoryManager())
    mem.add_message(user_message)

    short_term = list(mem.ephemeral)
    long_term  = mem.retrieve(user_message, k=5)

    prompt = "\n".join([
        "You are a helpful assistant.",
        "-- Ephemeral (last 3 messages):",
        *("  • " + m for m in short_term),
        "-- Long-term memories:",
        *("  • " + m for m in long_term),
        f"User: {user_message}",
        "Assistant:"
    ])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=5000,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, inputs.input_ids.shape[-1]:], skip_special_tokens=True)


In [11]:
import gradio as gr
from collections import deque
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load models once
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# MemoryManager class
class MemoryManager:
    def __init__(self, embed_model_name="all-MiniLM-L6-v2"):
        self.ephemeral = deque(maxlen=3)
        self.embed = SentenceTransformer(embed_model_name)
        dim = self.embed.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(dim)
        self.texts = []

    def add_message(self, text: str):
        self.ephemeral.append(text)
        emb = self.embed.encode(text, convert_to_numpy=True)
        self.index.add(np.array([emb], dtype="float32"))
        self.texts.append(text)

    def retrieve(self, query: str, k: int = 5):
        if not self.texts:
            return []
        q_emb = self.embed.encode(query, convert_to_numpy=True)
        D, I = self.index.search(np.array([q_emb], dtype="float32"), k)
        return [self.texts[i] for i in I[0] if i < len(self.texts)]

# Store sessions
session_memories = {}

# Core chat function
def chat(user_id, user_message):
    mem = session_memories.setdefault(user_id, MemoryManager())
    mem.add_message(user_message)

    short_term = list(mem.ephemeral)
    long_term = mem.retrieve(user_message, k=5)

    prompt = "\n".join([
        "You are a helpful assistant.",
        "-- Ephemeral (last 3 messages):",
        *("  • " + m for m in short_term),
        "-- Long-term memories:",
        *("  • " + m for m in long_term),
        f"User: {user_message}",
        "Assistant:"
    ])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(out[0, inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return reply

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Memory-Enhanced AI Chatbot")

    user_id = gr.Textbox(label="User ID", placeholder="Enter your name or ID")
    chat_input = gr.Textbox(label="Your Message", placeholder="Ask anything...", lines=2)
    chat_output = gr.Textbox(label="AI Assistant Reply", lines=4)

    send_button = gr.Button("Send")

    def handle_chat(uid, msg):
        if not uid or not msg:
            return "Please provide both user ID and message."
        return chat(uid.strip(), msg.strip())

    send_button.click(fn=handle_chat, inputs=[user_id, chat_input], outputs=chat_output)

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eddf09512f83864d98.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
